# Library 02: finding the batting partnerships

This is the heart of the project. Every edge in our networks comes from the partnerships found here, so if a
partnership starts or ends at the wrong ball, all later results are off. We kept the rules short and wrote
them down before writing the code.

**What we mean by a partnership:** the stretch of an innings in which the same two batters are at the crease
together. The pair has no order, so Root and Brook is the same pair as Brook and Root.

**Our rules**

1. We read the balls in the order they appear in the file: innings, then over, then ball.
2. Each ball adds its `runs.total` to the pair at the crease, exactly once. `runs.total` already contains the
   extras, so we never add the extras a second time.
3. When a ball has a wicket record for one of the two batters, the partnership ends **after** that ball. The
   runs scored on that ball still belong to the pair that was batting. We don't keep a list of "allowed"
   dismissal types: any wicket record naming a batter at the crease ends the partnership (this also covers
   retired hurt, retired out, obstructing the field and so on).
4. If the pair changes and there was no wicket, we only accept it when the file records a batter
   replacement. Anything else is written to the warnings table as `unexplained_pair_change`.
5. When the innings ends, the last partnership is closed once.
6. If two batters split up and later bat together again (for example when a retired-hurt batter comes back),
   that counts as two partnerships.
7. Penalty runs that Cricsheet gives to the whole innings don't belong to any ball, so we don't give them to
   any pair.

Needs `lib_00_settings` and `lib_01_data_loading`.

In [ ]:
@dataclass
class PartnershipObservation:
    """One partnership: the same two batters at the crease without a break."""

    match_id: str
    innings_index: int            # which innings of the match file (1 = first innings)
    partnership_seq: int          # 1 for the opening partnership, 2 for the next one, and so on
    player_1_id: str              # the two players are stored in ID order, so a pair always has the same key
    player_1_name: str
    player_2_id: str
    player_2_name: str
    partnership_runs: int = 0     # sum of runs.total over all balls of the partnership
    batter_runs: int = 0          # runs off the bat, only kept so we can check our totals
    extras_runs: int = 0          # extras, also only kept for checking
    deliveries: int = 0           # every ball record, wides and no-balls included
    start_delivery_id: str = ""
    end_delivery_id: str = ""
    boundary_reason: str = ""     # why it ended: dismissal:<kind>, retirement:<kind>, replacement or innings_end
    quality_status: str = "ok"    # set to 'warning' if we could not explain why the pair changed

    def as_row(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class InningsExtractionResult:
    """What we get back from one innings: the partnerships plus a few totals we use for checking."""

    observations: list[PartnershipObservation] = field(default_factory=list)
    warnings: list[dict[str, Any]] = field(default_factory=list)
    runs_from_deliveries: int = 0   # every runs.total in the innings; must equal the sum of the partnerships
    wicket_records: int = 0
    delivery_count: int = 0


def make_delivery_id(innings_index: int, over_number: int, ball_position: int) -> str:
    """Label for one ball: innings.over.position.

    We use the position of the ball inside the over (1, 2, 3, ...) and not the over.ball number printed in the
    file, because wides and no-balls make the printed numbers repeat (two balls can both be "12.4").
    """
    return f"{innings_index}.{over_number}.{ball_position}"


def is_retirement(wicket_kind: str) -> bool:
    """True for 'retired hurt', 'retired out' and the like."""
    return wicket_kind.lower().startswith("retired")


def delivery_replaces_batter(delivery: dict[str, Any], leaving_batters: set[str]) -> bool:
    """True if this ball records a replacement that takes one of the leaving batters off the field.

    Cricsheet keeps two kinds of replacement: 'role' (for example an injured batter swapped out) and 'match'
    (for example a concussion substitute). We accept both.
    """
    replacement_records = delivery.get("replacements", {}) or {}
    for role_change in replacement_records.get("role", []) or []:
        if role_change.get("role") == "batter" and role_change.get("out") in leaving_batters:
            return True
    for match_change in replacement_records.get("match", []) or []:
        if match_change.get("out") in leaving_batters:
            return True
    return False

## The innings extractor

The function goes through one innings ball by ball and remembers who the current pair is. The comments refer
back to the numbered rules above.

In [ ]:
def extract_innings_partnerships(innings_data: dict[str, Any], player_registry: dict[str, str],
                                 match_id: str, innings_index: int) -> InningsExtractionResult:
    """Find every partnership in one innings.

    player_registry is info.registry.people from the match file. It maps the name printed in the file to the
    player's fixed Cricsheet ID, so a player keeps the same node even if the name is spelled differently in some files.
    """
    result = InningsExtractionResult()

    def add_warning(warning_code: str, delivery_id: str, detail: str) -> None:
        result.warnings.append({"match_id": match_id, "innings_index": innings_index,
                                "delivery_id": delivery_id, "code": warning_code, "detail": detail})

    def player_id_of(player_name: str, delivery_id: str) -> str:
        if player_name in player_registry:
            return player_registry[player_name]
        # Edge case: a name that is missing from the registry. We don't guess an ID; we flag it instead.
        add_warning("player_not_in_registry", delivery_id, player_name)
        return f"unresolved:{player_name}"

    current_partnership: PartnershipObservation | None = None
    current_pair_names: set[str] = set()
    previous_delivery: dict[str, Any] | None = None
    batters_just_out: list[tuple[str, str]] = []   # (name, kind) of batters out on the previous ball

    def start_partnership(batter_name: str, non_striker_name: str, delivery_id: str) -> PartnershipObservation:
        # Sort the two players by ID, so Root/Brook and Brook/Root give the same pair.
        first_player, second_player = sorted([(player_id_of(batter_name, delivery_id), batter_name),
                                              (player_id_of(non_striker_name, delivery_id), non_striker_name)])
        return PartnershipObservation(
            match_id=match_id, innings_index=innings_index,
            partnership_seq=len(result.observations) + 1,
            player_1_id=first_player[0], player_1_name=first_player[1],
            player_2_id=second_player[0], player_2_name=second_player[1],
            start_delivery_id=delivery_id)

    def close_partnership(reason: str, status: str = "ok") -> None:
        nonlocal current_partnership, current_pair_names
        current_partnership.boundary_reason = reason
        if status != "ok":
            current_partnership.quality_status = status
        result.observations.append(current_partnership)
        current_partnership, current_pair_names = None, set()

    for over in innings_data.get("overs", []):
        over_number = over.get("over")
        # Edge case: an over with 7 or 5 balls (an umpire miscount). We simply follow the balls that are in
        # the file, so nothing special is needed here.
        for ball_position, delivery in enumerate(over.get("deliveries", []), start=1):
            delivery_id = make_delivery_id(innings_index, over_number, ball_position)
            pair_names_now = {delivery["batter"], delivery["non_striker"]}
            result.delivery_count += 1

            # Edge case: striker and non-striker are the same name. No pair can be formed, so we log it and
            # still count the runs for the innings total.
            if len(pair_names_now) != 2:
                add_warning("batter_equals_non_striker", delivery_id, delivery["batter"])
                result.runs_from_deliveries += int(delivery["runs"]["total"])
                continue

            # After a wicket ball we check that the batter who got out is really gone.
            if batters_just_out:
                for out_name, wicket_kind in batters_just_out:
                    if out_name in pair_names_now:
                        add_warning("player_out_still_at_crease", delivery_id, f"{out_name} ({wicket_kind})")
                batters_just_out = []

            # Rule 4: the pair is different but no wicket closed the old partnership.
            if current_partnership is not None and pair_names_now != current_pair_names:
                leaving_batters = current_pair_names - pair_names_now
                # The replacement can be recorded on this ball or on the ball before, so we look at both.
                replacement_recorded = delivery_replaces_batter(delivery, leaving_batters) or (
                    previous_delivery is not None and delivery_replaces_batter(previous_delivery, leaving_batters))
                if replacement_recorded:
                    close_partnership("replacement")
                else:
                    add_warning("unexplained_pair_change", delivery_id,
                                f"{sorted(current_pair_names)} -> {sorted(pair_names_now)}")
                    close_partnership("unexplained_pair_change", "warning")

            if current_partnership is None:
                current_partnership = start_partnership(delivery["batter"], delivery["non_striker"], delivery_id)
                current_pair_names = pair_names_now

            # Rule 2: add runs.total once. Batter runs and extras are only kept so we can check the sum later.
            delivery_runs = delivery["runs"]
            current_partnership.partnership_runs += int(delivery_runs["total"])
            current_partnership.batter_runs += int(delivery_runs.get("batter", 0))
            current_partnership.extras_runs += int(delivery_runs.get("extras", 0))
            current_partnership.deliveries += 1
            current_partnership.end_delivery_id = delivery_id
            result.runs_from_deliveries += int(delivery_runs["total"])

            # Rule 3: look at every wicket record on the ball.
            # Edge case: two wickets on one ball (say, a retirement and a run-out). Both batters leave together
            # and we do NOT create an in-between partnership that never faced a ball.
            wicket_records = delivery.get("wickets", []) or []
            result.wicket_records += len(wicket_records)
            batters_leaving = []
            for wicket in wicket_records:
                out_name, wicket_kind = wicket.get("player_out", ""), str(wicket.get("kind", "unknown"))
                if out_name in current_pair_names:
                    batters_leaving.append((out_name, wicket_kind))
                else:
                    # Edge case: the wicket names someone who is not batting. That would be a data problem.
                    add_warning("player_out_not_at_crease", delivery_id, f"{out_name} ({wicket_kind})")
            if batters_leaving:
                all_kinds = "+".join(kind for _, kind in batters_leaving)
                reason_type = "retirement" if all(is_retirement(kind) for _, kind in batters_leaving) else "dismissal"
                batters_just_out = batters_leaving
                close_partnership(f"{reason_type}:{all_kinds}")
            previous_delivery = delivery

    # Rule 5: close the last partnership when the innings ends.
    # Edge case: if the innings ended on a wicket ball, the partnership is already closed and nothing happens here.
    if current_partnership is not None:
        close_partnership("innings_end")
    return result

## From matches to tables

`extract_match_partnerships` runs the extractor on every **England** batting innings of one match.

Edge case: super overs. They are a tie-breaker, not part of the normal match, so we list them in the innings
table but take no partnerships from them.

In [ ]:
OBSERVATION_COLUMNS = [
    "match_id", "source_file", "match_date", "format", "innings_index", "batting_team",
    "partnership_seq", "player_1_id", "player_1_name", "player_2_id", "player_2_name",
    "partnership_runs", "batter_runs", "extras_runs", "deliveries",
    "start_delivery_id", "end_delivery_id", "boundary_reason", "quality_status",
]
WARNING_COLUMNS = ["match_id", "source_file", "format", "innings_index", "delivery_id", "code", "detail"]


def extract_match_partnerships(match_data: dict[str, Any], match_record: dict[str, Any]):
    """Return (partnership rows, innings rows, warning rows) for one match in the study."""
    player_registry = match_data["info"].get("registry", {}).get("people", {})
    observation_rows, innings_rows, warning_rows = [], [], []
    for innings_index, innings_data in enumerate(match_data.get("innings", []), start=1):
        if innings_data.get("team") != TEAM_NAME:
            continue   # we only study England's batting
        is_super_over = bool(innings_data.get("super_over", False))
        # Rule 7: penalty runs given to the whole innings are stored here, next to (not inside) the partnerships.
        penalty_runs = innings_data.get("penalty_runs", {}) or {}
        innings_row = {
            "match_id": match_record["match_id"], "format": match_record["format"],
            "match_date": match_record["start_date"], "innings_index": innings_index,
            "batting_team": innings_data.get("team"),
            "qualifying": not is_super_over, "super_over": is_super_over,
            "declared": bool(innings_data.get("declared", False)),
            "forfeited": bool(innings_data.get("forfeited", False)),
            "absent_hurt": "|".join(innings_data.get("absent_hurt", []) or []),
            "penalty_runs_pre": int(penalty_runs.get("pre", 0)),
            "penalty_runs_post": int(penalty_runs.get("post", 0)),
            "runs_from_deliveries": 0, "partnership_runs_sum": 0, "wicket_records": 0,
            "delivery_count": 0, "n_partnerships": 0,
        }
        if not is_super_over:
            innings_result = extract_innings_partnerships(innings_data, player_registry,
                                                          match_record["match_id"], innings_index)
            for observation in innings_result.observations:
                observation_row = observation.as_row()
                observation_row.update(format=match_record["format"], match_date=match_record["start_date"],
                                       source_file=match_record["filename"], batting_team=innings_data.get("team"))
                observation_rows.append(observation_row)
            for warning in innings_result.warnings:
                warning.update(format=match_record["format"], source_file=match_record["filename"])
                warning_rows.append(warning)
            innings_row.update(
                runs_from_deliveries=innings_result.runs_from_deliveries,
                partnership_runs_sum=sum(obs.partnership_runs for obs in innings_result.observations),
                wicket_records=innings_result.wicket_records,
                delivery_count=innings_result.delivery_count,
                n_partnerships=len(innings_result.observations))
        innings_rows.append(innings_row)
    return observation_rows, innings_rows, warning_rows


def extract_all_matches(included_matches: pd.DataFrame, match_folder: Path = RAW_MATCH_FOLDER):
    """Run the extractor on every match in the study. Returns the partnership, innings and warning tables."""
    all_observations, all_innings, all_warnings = [], [], []
    for match_record in included_matches.to_dict("records"):
        match_data = load_match_file(match_folder / match_record["filename"])
        observation_rows, innings_rows, warning_rows = extract_match_partnerships(match_data, match_record)
        all_observations += observation_rows
        all_innings += innings_rows
        all_warnings += warning_rows
    # We pass the column lists so that the tables keep the right columns even if they end up empty.
    return (pd.DataFrame(all_observations, columns=OBSERVATION_COLUMNS),
            pd.DataFrame(all_innings),
            pd.DataFrame(all_warnings, columns=WARNING_COLUMNS))